# SkyQuery E1: zero-shot prompt-based detectors on VisDrone
Pre-registered in the team's pre-registration notes (round 3). Runs every open-vocabulary model with two prompt sets on val, and with the class-name prompts on test-dev, then scores them against the fine-tuned YOLO26s with the same COCO harness and sequence-bootstrap intervals. Expect about 2–3 hours on 2×T4.

In [ ]:
# --- Setup: code, packages, data -------------------------------------------------
# 1) Upload skyquery.zip as a Kaggle Dataset named "skyquery-code" and add it to this notebook.
# 2) Settings -> Accelerator: GPU T4 x2, Internet: on.
# 3) Optional (OWLv2 / Grounding DINO): Add-ons -> Secrets -> HF_TOKEN (read-only token).
import os, subprocess, sys, glob, shutil
CODE = glob.glob("/kaggle/input/skyquery-code/**/pyproject.toml", recursive=True)
assert CODE, "Add the skyquery-code dataset to this notebook"
SRC = os.path.dirname(CODE[0])
WORK = "/kaggle/working/skyquery"
if not os.path.exists(WORK):
    shutil.copytree(SRC, WORK, ignore=shutil.ignore_patterns("node_modules", "data", "cache"))
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[eval]", "transformers>=4.57", "git+https://github.com/ultralytics/CLIP.git"], check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret: OWLv2 / Grounding DINO will try anonymous downloads")

os.makedirs("data", exist_ok=True); os.makedirs("cache", exist_ok=True)
BASE = "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
def fetch(split):
    d = f"data/VisDrone2019-DET-{split}"
    if not os.path.exists(d):
        subprocess.run(f"curl -sSL -o /tmp/{split}.zip {BASE}VisDrone2019-DET-{split}.zip && unzip -q /tmp/{split}.zip -d data && rm /tmp/{split}.zip", shell=True, check=True)
    return d
VAL, TEST = fetch("val"), fetch("test-dev")


In [ ]:
# Fine-tuned reference (FI1280) on GPU, cached in the same format
for tag, split in (("val", VAL), ("testdev", TEST)):
    subprocess.run([sys.executable, "-m", "skyquery.calibrate.cache_preds", "--split", split, "--scale", "1280",
                    "--device", "0", "--out", f"cache/{tag}_1280.npz"], check=True)


In [ ]:
MODELS = ["yoloe-26s", "yoloe-26m", "yoloe-26l", "yolo-world-v2-s", "yolo-world-v2-l",
          "owlv2-base", "owlv2-large", "grounding-dino-tiny", "grounding-dino-base"]
def run(model, prompts, tag, split):
    out = f"cache/{tag}_{model}_{prompts}.npz"
    dev = "0" if model.startswith(("yolo")) else "cuda"
    r = subprocess.run([sys.executable, "-m", "skyquery.calibrate.cache_ovd", "--split", split, "--model", model,
                        "--prompts", prompts, "--device", dev, "--out", out])
    return out if r.returncode == 0 else None

done = {"val": [], "testdev": []}
for m in MODELS:
    for p in ("names", "described"):
        o = run(m, p, "val", VAL)
        if o: done["val"].append(f"{m}:{p}={o}")
    o = run(m, "names", "testdev", TEST)
    if o: done["testdev"].append(f"{m}:names={o}")
print(done)


In [ ]:
for tag, split in (("val", VAL), ("testdev", TEST)):
    subprocess.run([sys.executable, "-m", "skyquery.calibrate.ovd_eval", "e1", "--split", split, "--runs", *done[tag]], check=True)


In [ ]:
# Send these back: reports/ovd_e1_*.json (small) and, if you can, the caches
subprocess.run("zip -qr /kaggle/working/e1_results.zip reports cache -i 'reports/*.json' 'cache/*.npz'", shell=True)
print("Download /kaggle/working/e1_results.zip from the Output panel")
